# Implementación del algoritmo de optimización heurístico ABC (Artificial Bee Colony)

## Parámetros de control del algoritmo

- **SN**: número de fuentes de alimento (soluciones).
- **MCN**: número máximo de ciclos de iteración.
- **limit**: número máximo de intentos de mejora de una solución.
- **f**: función objetivo.

In [1]:
# Librerías necesarias
from __future__ import annotations
from typing import Callable
import pandas as pd
import numpy as np
import pandas as pd
from IPython.display import display

## Implementación principal

In [2]:
class ArtificialBeeColony:
    # Atributos
    limit: int
    sn: int
    mcn: int
    f: Callable
    seed: int # Semilla de aleatorización.
    history: dict # Para guardar el historial

    def __init__(self, limit: int, sn: int, mcn: int, f: Callable, seed: int) -> None:
        self.limit = limit
        self.sn = sn
        self.mcn = mcn
        self.f = f
        self.seed = seed
        self.history = {}

    def execute(self, restrictions: list[tuple]):

        if self.sn < 2:
            raise Exception("Debe haber mínimo dos soluciones.")
            
        self.history = {}
        rng = np.random.default_rng(self.seed)
        solutions: list = [None]*self.sn
        trial : list = [0]*self.sn # Contador de intentos

        # Inicialización de los vectores de solución.
        rows = []
        for i in range(self.sn):
            solutions[i] = [self.generate_random_solution(rng, lower, upper) for lower, upper in restrictions]
            rows.append({"Solución": i+1, 
                        "Componentes": solutions[i].copy(), 
                        "Valor en función objetivo": self.f(*solutions[i]),
                        })

        self.history["inicializacion"] = rows

        mat_solutions = np.array(solutions)
        best_solution = None
        best_value = None

        # Inicio del algoritmo
        for cyc in range(self.mcn):
            # Fase de abejas obreras
            rows = []
            replaced = [False]*self.sn
            for i in range(self.sn):
                
                new_solution, k, j, phi = self.generate_neighbor_solution(rng, mat_solutions, restrictions, i)
                rows.append({
                    "Ciclo": cyc + 1,
                    f"Solución Nueva": i+1,
                    "k": k,
                    "j": j,
                    "phi": phi,
                    "Componentes": new_solution.copy(),
                })
                
                if self.f(*new_solution) < self.f(*mat_solutions[i]):
                    mat_solutions[i] = new_solution
                    trial[i] = 0
                    replaced[i] = True 
                else:
                    trial[i] += 1

            self.history.setdefault("candidatas_fase_obreras", []).extend(rows)

            # Intermedio, se contruye la tabla de reemplazos de esta fase
            rows = []
            for i in range(self.sn):
                rows.append({
                    "Ciclo": cyc + 1,
                    "Solución": i + 1,
                    "Reemplazada": replaced[i],
                    "Componentes": mat_solutions[i].copy(),
                    "Valor en función objetivo": self.f(*mat_solutions[i]),
                    "Trial": trial[i],
                })

            self.history.setdefault("reemplazadas_fase_obreras", []).extend(rows)

            # Fase de abejas observadoras
            fitness = [self.fitness(mat_solutions, i) for i in range(self.sn)]
            max_fitness = max(fitness)
            p = [self.calculate_probability(fitness[i], max_fitness) for i in range(self.sn)]

            i = 0
            t = 0
            rows = []
            replaced = [False]*self.sn
            while t < self.sn:
                r = rng.uniform(0, 1)
                record = {"Ciclo": cyc + 1, "Iteración (Solución evaluada)": i + 1, "Probabilidad": p[i], "r": r, "r < p": r < p[i]}
                if r < p[i]:
                    t += 1
                    new_solution, k, j, phi = self.generate_neighbor_solution(rng, mat_solutions, restrictions, i)
                    record.update({"Nuevo candidato": new_solution.copy(), 
                              "Valor en función objetivo": self.f(*new_solution)})
                    if self.f(*new_solution) < self.f(*mat_solutions[i]):
                        mat_solutions[i] = new_solution
                        trial[i] = 0
                        record.update({"Reemplazada": True, "Trial": 0})
                        replaced[i] = True
                    else:
                        trial[i] += 1
                        record.update({"Reemplazada": False, "Trial": trial[i]})

                record.update({"Trial": trial[i]})
                rows.append(record)

                i = (i + 1) % self.sn

            self.history.setdefault("fase_abejas_observadoras", []).extend(rows)

            # Intermedio, se contruye la tabla de reemplazos de esta fase
            rows = []
            for i in range(self.sn):
                rows.append({
                    "Ciclo": cyc + 1,
                    "Solución": i + 1,
                    "Reemplazada": replaced[i],
                    "Componentes": mat_solutions[i].copy(),
                    "Valor en función objetivo": self.f(*mat_solutions[i]),
                    "Trial": trial[i],
                })

            self.history.setdefault("fase_abejas_observadoras_resumen", []).extend(rows)

            # Fase de abejas exploradoras

            values = np.array([self.f(*solution) for solution in mat_solutions])
            best_index = np.argmin(values)
            if best_value is None or values[best_index] < best_value:
                best_solution = mat_solutions[best_index].copy()
                best_value = values[best_index]


            replaced = [False]*self.sn
            for i, num_trial in enumerate(trial):
                if num_trial > self.limit:
                    mat_solutions[i] = np.array([
                        self.generate_random_solution(rng, lower, upper) for lower, upper in restrictions])
                    trial[i] = 0
                    replaced[i] = True

            # Registro del resultado de la fase exploradora
            rows = []
            for i in range(self.sn):
                rows.append({
                    "Ciclo": cyc + 1,
                    "Solución": i + 1,
                    "Reemplazada": replaced[i],
                    "Componentes": mat_solutions[i].copy(),
                    "Valor en función objetivo": self.f(*mat_solutions[i]),
                    "Trial": trial[i],
                })

            self.history.setdefault("fase_abejas_exploradoras", []).extend(rows)

        return best_solution, best_value, self.history

    def generate_random_solution(self, rng, lower_bound: float, upper_bound : float) -> float:
        return lower_bound + rng.uniform(0, 1)*(upper_bound - lower_bound)

    def generate_neighbor_solution(self, rng, solutions, restrictions: list[tuple], i: int) -> tuple:
    
        j = rng.choice(np.arange(len(solutions[i])))
        restriction_lower, restriction_upper = restrictions[j]

        new_solution = solutions[i].copy()

        phi = None
        k = None
        new_value = None
        while True:
            phi = rng.uniform(-1, 1)
            indexes = np.arange(len(solutions))
            valid_indexes = indexes[indexes != i]
            k = rng.choice(valid_indexes)
            new_value = solutions[i, j] + phi*(solutions[i, j] - solutions[k, j])
            if restriction_lower <= new_value <= restriction_upper:
                break

        new_solution[j] = new_value

        return new_solution, k, j, phi
        
    def fitness(self, solutions, i : int) -> float:
        f = self.f(*solutions[i])
        if f >= 0:
            return 1/(1 + f)
        else:
            return 1 + abs(f)

    def calculate_probability(self, fitness_i: float, max_fitness: float) -> float:
        return 0.1 + 0.9*(fitness_i / max_fitness)
        


In [3]:
def show_history(history: dict) -> None:
    # Formatear los números decimales a tres cifras.
    pd.options.display.float_format = '{:.3f}'.format

    # Mostrar la inicialización.
    print("\n" + "=" * 70)
    print("INICIALIZACIÓN")
    print("=" * 70)

    df = pd.DataFrame(history["inicializacion"])
    display(format_dataframe(df).style.hide(axis="index"))

    # Identificar las fases iterativas.
    phases = [
        phase for phase in history
        if phase != "inicializacion"
    ]

    # Obtener el número total de ciclos.
    mcn = max(
        row["Ciclo"]
        for phase in phases
        for row in history[phase]
    )

    # Mostrar las fases de cada ciclo.
    for cyc in range(1, mcn + 1):
        print(f"\n{'#' * 70}")
        print(f"CICLO {cyc}")
        print(f"{'#' * 70}")

        for phase in phases:
            cycle_rows = [
                row for row in history[phase]
                if row["Ciclo"] == cyc
            ]

            print(f"\n{phase.upper().replace('_', ' ')}")

            df = pd.DataFrame(cycle_rows)
            display(format_dataframe(df).style.hide(axis="index"))


def format_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    # Trabajar sobre una copia para no modificar el DataFrame original.
    df = df.copy()

    # Redondear los componentes de los vectores a tres cifras.
    for column in ["Componentes", "Nuevo candidato"]:
        if column in df.columns:
            df[column] = df[column].map(
                lambda vector: np.round(vector, 3)
                if isinstance(vector, (list, np.ndarray))
                else vector
            )

    return df

In [4]:
f = lambda x1, x2, x3, x4, x5: x1**2 + x2**2 + x3**2 + x4**2 + x5**2
algorithm = ArtificialBeeColony(2, 5, 10, f, 42)
restrictions = [(-5.12, 5.12), (-5.12, 5.12), (-5.12, 5.12), (-5.12, 5.12), (-5.12, 5.12)]
best_solution, best_value, history = algorithm.execute(restrictions)

In [ ]:
print("Mejor valor: ", best_value)
print("Mejor solución: ", best_solution)
show_history(algorithm.history)

Mejor valor:  3.7632653852504454
Mejor solución:  [ 1.68285803  0.44375132 -0.61581245  0.40825795 -0.43409615]

INICIALIZACIÓN


Solución,Componentes,Valor en función objetivo
1,[ 2.805 -0.626 3.672 2.021 -4.156],43.099241
2,[ 4.87 2.674 2.929 -3.808 -0.508],54.211838
3,[-1.323 4.37 1.473 3.305 -0.579],34.277498
4,[-2.793 0.559 -4.467 3.355 1.348],41.136821
5,[ 2.643 -1.49 4.82 4.026 2.851],56.766779



######################################################################
CICLO 1
######################################################################

CANDIDATAS FASE OBRERAS


Ciclo,Solución Nueva,k,j,phi,Componentes
1,1,1,3,-0.066558,[ 2.805 -0.626 3.672 1.633 -4.156]
1,2,0,2,-0.691421,[ 4.87 2.674 3.443 -3.808 -0.508]
1,3,3,3,0.489524,[-1.323 4.37 1.473 3.281 -0.579]
1,4,4,1,-0.348349,[-2.793 -0.155 -4.467 3.355 1.348]
1,5,1,4,-0.060888,[ 2.643 -1.49 4.82 4.026 2.646]



REEMPLAZADAS FASE OBRERAS


Ciclo,Solución,Reemplazada,Componentes,Valor en función objetivo,Trial
1,1,True,[ 2.805 -0.626 3.672 1.633 -4.156],41.681525,0
1,2,False,[ 4.87 2.674 2.929 -3.808 -0.508],54.211838,1
1,3,True,[-1.323 4.37 1.473 3.281 -0.579],34.116741,0
1,4,True,[-2.793 -0.155 -4.467 3.355 1.348],40.848324,0
1,5,True,[ 2.643 -1.49 4.82 4.026 2.646],55.642656,0


... (92 outputs left out)


FASE ABEJAS OBSERVADORAS RESUMEN


Ciclo,Solución,Reemplazada,Componentes,Valor en función objetivo,Trial
10,1,False,[ 0.783 -2.768 0.667 1.329 -0.304],10.579091,1
10,2,True,[ 1.683 0.444 -0.616 0.408 -0.434],3.763265,0
10,3,False,[-2.566 0.729 -0.499 -4.616 -1.294],30.346038,2
10,4,False,[ 3.763 0.18 -2.582 -1.626 2.399],29.260726,1
10,5,False,[ 0.243 -4.079 3.415 -2.081 4.35 ],51.612424,0



FASE ABEJAS EXPLORADORAS


Ciclo,Solución,Reemplazada,Componentes,Valor en función objetivo,Trial
10,1,False,[ 0.783 -2.768 0.667 1.329 -0.304],10.579091,1
10,2,False,[ 1.683 0.444 -0.616 0.408 -0.434],3.763265,0
10,3,False,[-2.566 0.729 -0.499 -4.616 -1.294],30.346038,2
10,4,False,[ 3.763 0.18 -2.582 -1.626 2.399],29.260726,1
10,5,False,[ 0.243 -4.079 3.415 -2.081 4.35 ],51.612424,0
